# BirdCLEF V18 CV Gate Runner

Runs V18 in `train` mode, saves OOF/SED artifacts, then runs the CV gate for `w_proto=0.65` vs `w_proto=0.60`.

Kaggle setup: add the BirdCLEF 2026 competition dataset as an input. Enable Internet if cloning from GitHub inside the notebook.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/kaggle/working/BirdCLEF')
BRANCH = 'codex/experiment-001'

if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/hinemos-anzu/BirdCLEF.git {REPO_DIR}
else:
    print(f'Repository already exists: {REPO_DIR}')

os.chdir('/kaggle/working')
print('cwd =', Path.cwd())
print('repo =', REPO_DIR)


In [ ]:
BASE = Path('/kaggle/input/competitions/birdclef-2026')
print('BASE exists:', BASE.exists())
print('train_soundscapes:', len(list((BASE / 'train_soundscapes').glob('*.ogg'))) if (BASE / 'train_soundscapes').exists() else 'missing')
print('test_soundscapes:', len(list((BASE / 'test_soundscapes').glob('*.ogg'))) if (BASE / 'test_soundscapes').exists() else 'missing')
print('sample_submission exists:', (BASE / 'sample_submission.csv').exists())
print('labels exists:', (BASE / 'train_soundscapes_labels.csv').exists())


In [ ]:
# Force V18 to train mode for local OOF/CV generation.
# Safe when MODE is already train. Edits only the Kaggle working copy.
from pathlib import Path
import re

script_path = REPO_DIR / 'birdclef_v18_full_pipeline_oof_codex_ready_fixed.py'
text = script_path.read_text(encoding='utf-8')
pattern = r'(?m)^(\s*)MODE\s*=\s*["\'](?:train|submit)["\'](.*)$'
match = re.search(pattern, text)
if match is None:
    candidates = [line for line in text.splitlines() if 'MODE' in line][:20]
    raise RuntimeError('MODE assignment was not found. Candidate lines: ' + repr(candidates))

text2 = re.sub(pattern, r'\1MODE = "train"\2', text, count=1)
script_path.write_text(text2, encoding='utf-8')

for i, line in enumerate(script_path.read_text(encoding='utf-8').splitlines(), 1):
    if re.match(r'\s*MODE\s*=', line):
        print(f'{i}: {line}')
        break


## Run V18

This can take a while. `%run` keeps variables such as `sed_preds_tr_aligned`, `oof_proto_probs`, `Y_FULL_aligned`, and `meta_tr` available after the script finishes.


In [ ]:
%run /kaggle/working/BirdCLEF/birdclef_v18_full_pipeline_oof_codex_ready_fixed.py


In [ ]:
import numpy as np
from pathlib import Path

cache_dir = Path('/kaggle/working/cache')
cache_dir.mkdir(parents=True, exist_ok=True)

required_vars = ['sed_preds_tr_aligned', 'oof_proto_probs', 'Y_FULL_aligned', 'meta_tr']
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(f'Missing expected variables after V18 run: {missing}')

np.save(cache_dir / 'sed_preds_tr_aligned.npy', sed_preds_tr_aligned)
np.save(cache_dir / 'oof_proto_probs.npy', oof_proto_probs)
meta_tr.to_parquet(cache_dir / 'perch_meta.parquet')

print('saved:', cache_dir / 'sed_preds_tr_aligned.npy', sed_preds_tr_aligned.shape)
print('saved:', cache_dir / 'oof_proto_probs.npy', oof_proto_probs.shape)
print('saved:', cache_dir / 'perch_meta.parquet', meta_tr.shape)


In [ ]:
!python /kaggle/working/BirdCLEF/codex-experiments/001/cv_blend_gate_runner.py \
  --sed-oof /kaggle/working/cache/sed_preds_tr_aligned.npy \
  --candidate-w 0.65 \
  --baseline-w 0.60


In [ ]:
import json
import pandas as pd
from pathlib import Path

out_dir = Path('/kaggle/working/codex_experiment_001')
table = pd.read_csv(out_dir / 'blend_cv_gate_table.csv')
decision = json.loads((out_dir / 'blend_cv_gate_decision.json').read_text())
display(table)
decision


## Decision Rule

Submit `w_proto=0.65` only when `should_submit` is `true`. Otherwise keep `w_proto=0.60` or move to the next improvement candidate.
